# CPU-only Receipt & Invoice Parser
**Google Colab notebook | Retail receipts + business invoices | OCR + rules + small classifier**

This prototype:
- accepts JPG, PNG, TIFF, and PDF files;
- renders PDF pages and preprocesses images;
- uses Tesseract OCR to collect text, bounding boxes, and confidence scores;
- extracts vendor, invoice/receipt number, date, currency, subtotal, tax, total, GSTIN, and line items;
- uses a small TF-IDF + logistic-regression classifier as a supporting signal;
- runs validation checks and exports JSON, CSV, and Excel.

**Important:** this is a prototype, not accounting software. OCR and extraction can be wrong. Review flagged documents and validate against originals before using results for payments, tax filing, or bookkeeping. The classifier is only as good as its labeled examples; rules remain the fallback.

## Cell 1 — Install dependencies

Run this cell once per fresh Colab runtime. Tesseract is installed as a system package; Python libraries are installed with pip.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr
!pip -q install pytesseract opencv-python-headless pymupdf pillow pandas scikit-learn openpyxl

## Cell 2 — Imports and configuration

Set `OCR_LANGUAGE` to `eng` for English documents. If you need additional languages, install the relevant Tesseract language packs and update this setting.

In [ ]:
import os, re, json, math, shutil, warnings
from pathlib import Path
from datetime import datetime

import cv2
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import pytesseract

from PIL import Image, ImageOps, ImageEnhance
from pytesseract import Output
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.base import clone

warnings.filterwarnings("ignore")

OCR_LANGUAGE = "eng"
DPI = 220                  # PDF rendering resolution
MAX_IMAGE_SIDE = 2400      # keep CPU/RAM use reasonable
PREPROCESS_FOR_OCR = True
OUTPUT_DIR = Path("/content/receipt_invoice_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Tesseract:", pytesseract.get_tesseract_version())
print("Output directory:", OUTPUT_DIR)

## Cell 3 — Upload documents

Upload any combination of image files and PDFs. You can also skip the upload and use the synthetic example later. Uploaded files are available only in the current Colab runtime unless you download or save them elsewhere.

In [ ]:
from google.colab import files

uploaded = files.upload()
INPUT_DIR = Path("/content/receipt_invoice_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, data in uploaded.items():
    safe_name = Path(name).name
    (INPUT_DIR / safe_name).write_bytes(data)

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".pdf"}
input_files = sorted(
    p for p in INPUT_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
)
print(f"Found {len(input_files)} supported file(s).")
for p in input_files:
    print(" •", p.name)

## Cell 4 — Convert images and PDFs into page images

Every PDF page is rendered to an image, so scanned PDFs and ordinary PDFs follow the same OCR path. For large PDFs, consider limiting the page range to control runtime.

In [ ]:
def load_document_pages(path, dpi=DPI):
    """Return a list of (page_label, PIL.Image) for an image or PDF."""
    path = Path(path)
    ext = path.suffix.lower()

    if ext == ".pdf":
        pages = []
        doc = fitz.open(path)
        zoom = dpi / 72.0
        matrix = fitz.Matrix(zoom, zoom)
        for i, page in enumerate(doc):
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            pages.append((f"{path.stem}_page_{i+1}", image))
        doc.close()
        return pages

    image = Image.open(path).convert("RGB")
    return [(path.stem, image)]

if input_files:
    print("Example page counts:")
    for p in input_files:
        print(p.name, "→", len(load_document_pages(p)), "page(s)")
else:
    print("No files uploaded yet. You can still run the sample test cells below.")

## Cell 5 — Image preprocessing

The function corrects EXIF orientation, scales oversized images down, and improves contrast. Aggressive thresholding can damage faint thermal receipts, so the OCR function can compare grayscale and thresholded versions when needed.

In [ ]:
def preprocess_image(pil_image, max_side=MAX_IMAGE_SIDE):
    image = ImageOps.exif_transpose(pil_image).convert("RGB")
    w, h = image.size
    scale = min(1.0, max_side / max(w, h))
    if scale < 1.0:
        image = image.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)

    arr = np.array(image)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    # Mild denoising + contrast normalization; avoid overly aggressive morphology.
    gray = cv2.fastNlMeansDenoising(gray, None, h=8, templateWindowSize=7, searchWindowSize=21)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return image, enhanced

# Quick visual check on the first uploaded page, if present.
if input_files:
    preview_pages = load_document_pages(input_files[0])
    original_preview, processed_preview = preprocess_image(preview_pages[0][1])
    display(original_preview)
    display(Image.fromarray(processed_preview))

## Cell 6 — OCR with word positions and confidence

The OCR result contains word-level text, confidence, and bounding boxes. These coordinates help group words into lines and can later support a more advanced layout-aware model.

In [ ]:
def ocr_page(pil_image, language=OCR_LANGUAGE, preprocess=True):
    original, enhanced = preprocess_image(pil_image)
    if preprocess:
        ocr_image = enhanced
    else:
        ocr_image = np.array(original)

    data = pytesseract.image_to_data(
        ocr_image,
        lang=language,
        config="--oem 3 --psm 6",
        output_type=Output.DATAFRAME
    )
    data = data.dropna(subset=["text"]).copy()
    data["text"] = data["text"].astype(str).str.strip()
    data["conf"] = pd.to_numeric(data["conf"], errors="coerce")
    data = data[(data["text"] != "") & (data["conf"] >= 0)].copy()

    if data.empty:
        return pd.DataFrame(columns=["text", "conf", "left", "top", "width", "height", "line_num", "block_num"])

    data["right"] = data["left"] + data["width"]
    data["bottom"] = data["top"] + data["height"]
    return data.reset_index(drop=True)

def words_to_lines(word_df, y_tolerance=10):
    """Group OCR words by Tesseract's block/paragraph/line identifiers."""
    if word_df.empty:
        return []
    keys = [c for c in ["block_num", "par_num", "line_num"] if c in word_df.columns]
    lines = []
    for _, group in word_df.groupby(keys, sort=True):
        group = group.sort_values("left")
        text = " ".join(group["text"].astype(str))
        lines.append({
            "text": text,
            "conf": float(group["conf"].mean()),
            "left": int(group["left"].min()),
            "top": int(group["top"].min()),
            "right": int(group["right"].max()),
            "bottom": int(group["bottom"].max()),
            "words": group["text"].astype(str).tolist()
        })
    return sorted(lines, key=lambda x: (x["top"], x["left"]))

if input_files:
    test_pages = load_document_pages(input_files[0])
    test_words = ocr_page(test_pages[0][1])
    test_lines = words_to_lines(test_words)
    print("\n".join(line["text"] for line in test_lines[:40]))
else:
    print("Upload a document in Cell 3 to test OCR.")

## Cell 7 — Small CPU classifier

This classifier predicts a coarse label for each OCR line: vendor, document number, date, subtotal, tax, total, line item, or other. The seed examples are deliberately small and only demonstrate the pipeline; add representative labeled examples from your own documents to improve it.

Do not treat the classifier's prediction as ground truth. The extraction rules below remain the main mechanism for field values.

In [ ]:
TRAINING_EXAMPLES = [
    # Vendor / merchant
    ("ABC OFFICE SUPPLIES", "vendor"),
    ("Fresh Mart Supermarket", "vendor"),
    ("Tata Retail Enterprise", "vendor"),
    ("XYZ TRADERS PRIVATE LIMITED", "vendor"),
    ("INVOICE", "other"),
    # Document number
    ("Invoice No: INV-2026-1042", "document_number"),
    ("Receipt No 004512", "document_number"),
    ("Bill Number: 87219", "document_number"),
    ("Tax Invoice No. AB/2026/018", "document_number"),
    # Dates
    ("Date: 12/10/2026", "date"),
    ("Invoice Date 2026-10-12", "date"),
    ("12 Oct 2026", "date"),
    ("Date of issue: 10/12/2026", "date"),
    # Subtotal
    ("Subtotal 1,000.00", "subtotal"),
    ("Net amount 1000.00", "subtotal"),
    ("Taxable value 1000.00", "subtotal"),
    # Tax
    ("GST 180.00", "tax"),
    ("CGST 90.00 SGST 90.00", "tax"),
    ("IGST 180.00", "tax"),
    ("VAT 20.00", "tax"),
    # Total
    ("Grand Total INR 1,180.00", "total"),
    ("Amount Due 1180.00", "total"),
    ("TOTAL Rs. 1180.00", "total"),
    ("Balance Payable 1180.00", "total"),
    # Line items
    ("Printer paper A4 2 500.00 1000.00", "line_item"),
    ("Milk 1L 2 60.00 120.00", "line_item"),
    ("USB cable 1 250.00 250.00", "line_item"),
    ("Notebook 3 40.00 120.00", "line_item"),
    # Other
    ("Thank you for shopping", "other"),
    ("Goods once sold will not be taken back", "other"),
    ("Payment by card", "other"),
    ("GSTIN: 22AAAAA0000A1Z5", "other"),
]

classifier = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), lowercase=True, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])
classifier.fit([x for x, y in TRAINING_EXAMPLES], [y for x, y in TRAINING_EXAMPLES])

def classify_lines(lines):
    if not lines:
        return []
    texts = [line["text"] for line in lines]
    predictions = classifier.predict(texts)
    probabilities = classifier.predict_proba(texts).max(axis=1)
    result = []
    for line, label, prob in zip(lines, predictions, probabilities):
        item = dict(line)
        item["classifier_label"] = str(label)
        item["classifier_confidence"] = float(prob)
        result.append(item)
    return result

print("Classifier trained on", len(TRAINING_EXAMPLES), "starter examples.")
print("For a real deployment, label more examples from your own receipt/invoice formats.")

## Cell 8 — Field extraction rules

This parser uses labeled OCR lines and regular expressions. Currency and number conventions vary between countries; this starter parser is tuned primarily for common English-language invoices and Indian GST documents. Review extracted values, especially ambiguous dates and decimal separators.

In [ ]:
# Safer amount/date/document-number rules.
# Key change: bare integers are NOT treated as money. Amounts should have decimals,
# currency symbols/codes, or occur on an explicitly labeled financial line.
DATE_PATTERNS = [
    re.compile(r'\b(\d{4}[-/]\d{1,2}[-/]\d{1,2})\b'),
    re.compile(r'\b(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})\b'),
    re.compile(r'\b(\d{1,2}\s+[A-Za-z]{3,9}\s+\d{2,4})\b'),
    re.compile(r'\b([A-Za-z]{3,9}\s+\d{1,2},?\s+\d{2,4})\b')
]
GSTIN_RE = re.compile(r'\b\d{2}[A-Z]{5}\d{4}[A-Z][A-Z0-9]Z[A-Z0-9]\b', re.I)

# Explicit currency tokens or decimal amounts; bare integers such as street numbers,
# ZIP codes, phone numbers, and date components are intentionally ignored.
MONEY_TOKEN_RE = re.compile(
    r'(?<![A-Za-z0-9])(?:₹|\$|€|£|\bINR\b|\bUSD\b|\bEUR\b|\bGBP\b|\bRs\.?\s*)\s*'
    r'\(?-?\d[\d,]*(?:\.\d{1,2})?\)?'
    r'|(?<![A-Za-z0-9])\(?-?\d{1,3}(?:,\d{3})+(?:\.\d{1,2})?\)?'
    r'|(?<![A-Za-z0-9])\(?-?\d+\.\d{1,2}\)?'
    , re.I)

FINANCIAL_LABEL_RE = re.compile(
    r'(?i)\b(?:grand\s*total|amount\s*due|balance\s*due|amount\s*payable|'
    r'sub\s*total|subtotal|net\s*amount|taxable\s*value|total|'
    r'cgst|sgst|igst|gst|vat|sales\s*tax|discount|shipping|delivery|change|cash)\b'
)

EXCLUDE_ITEM_RE = re.compile(
    r'(?i)\b(?:invoice|receipt|bill\s*(?:no|number|#)?|date|received|due date|'
    r'payment method|payment|cashier|customer|reminders?|contact us|email|phone|tel|'
    r'www\.|https?://|gstin|subtotal|sub\s*total|grand\s*total|amount due|'
    r'balance due|amount payable|taxable value|total|cgst|sgst|igst|gst|vat|tax|'
    r'change|round off|thank you|terms|sold|return policy)\b'
)
ADDRESS_HINT_RE = re.compile(r'(?i)\b(?:street|st\.?|road|rd\.?|drive|dr\.?|avenue|ave\.?|suite|parkway|pkwy|zip)\b')


def normalize_space(text):
    return re.sub(r'\s+', ' ', str(text)).strip()


def parse_money_value(raw):
    if raw is None:
        return None
    s = str(raw).strip()
    s = re.sub(r'(?i)\b(?:INR|USD|EUR|GBP|Rs)\.?\b', '', s)
    s = s.replace('₹', '').replace('$', '').replace('€', '').replace('£', '')
    s = s.replace(',', '').replace(' ', '').replace('(', '-').replace(')', '')
    s = re.sub(r'[^0-9.\-]', '', s)
    if not s or s in {'-', '.', '-.'}:
        return None
    try:
        return round(float(s), 2)
    except ValueError:
        return None


def amounts_in_line(text, allow_bare_integer=False):
    """Return monetary candidates; by default, ignore bare integers."""
    text = str(text)
    candidates = []
    for match in MONEY_TOKEN_RE.finditer(text):
        raw = match.group(0).strip()
        value = parse_money_value(raw)
        if value is not None:
            candidates.append((raw, value))
    # For explicitly labeled financial lines, allow a bare integer such as 'TOTAL 9'.
    if allow_bare_integer and FINANCIAL_LABEL_RE.search(text) and not candidates:
        matches = re.findall(r'(?<![A-Za-z0-9])\d[\d,]*(?![A-Za-z0-9])', text)
        if matches:
            value = parse_money_value(matches[-1])
            if value is not None:
                candidates.append((matches[-1], value))
    return candidates


def find_date(text):
    for pattern in DATE_PATTERNS:
        m = pattern.search(str(text))
        if m:
            return m.group(1)
    return None


def extract_document_number(lines):
    """Extract IDs when the label and value are on the same or adjacent OCR lines."""
    label_re = re.compile(r'(?i)\b(?:invoice|receipt|bill|document)\s*(?:no\.?|number|num\.?|#)\s*[:#-]?\s*(.*)$')
    short_label_re = re.compile(r'(?i)^\s*(?:invoice|receipt|bill|document)\s*(?:no\.?|number|num\.?|#)\s*[:#-]?\s*$')
    prefix_re = re.compile(r'(?i)\b(?:inv|rcpt)[-/ ]+([A-Z0-9][A-Z0-9\-/]{2,})\b')
    # IDs may be long numeric strings (e.g. 5177183057228876365), so don't impose a short max length.
    standalone_id_re = re.compile(r'^\s*([A-Z0-9][A-Z0-9/-]{2,})\s*$')

    ordered = sorted(lines, key=lambda x: (x.get('page', 1), x.get('top', 0)))
    for i, line in enumerate(ordered):
        text = normalize_space(line.get('text', ''))
        m = label_re.search(text)
        if m:
            same_line_value = normalize_space(m.group(1)).strip(' :-#')
            if same_line_value:
                # Use the first token after the label, excluding punctuation.
                token_match = re.search(r'([A-Z0-9][A-Z0-9/-]{2,})', same_line_value, re.I)
                if token_match:
                    value = token_match.group(1).strip(' :-#')
                    if value.lower() not in {'date', 'number', 'total', 'invoice', 'receipt'}:
                        return value
            # The label may be on its own line and the identifier immediately below it.
            for following in ordered[i+1:i+3]:
                candidate = normalize_space(following.get('text', ''))
                # Stop if another label starts before an ID is found.
                if re.search(r'(?i)\b(?:date|total|subtotal|tax|gstin|payment|due date)\b', candidate):
                    break
                id_match = standalone_id_re.match(candidate)
                if id_match:
                    value = id_match.group(1)
                    if not find_date(candidate) and not GSTIN_RE.fullmatch(value):
                        return value
        m = prefix_re.search(text)
        if m:
            return m.group(1).strip(' :-#')
    return None

def detect_currency(full_text):
    if '₹' in full_text or re.search(r'\bINR\b|\bRs\.?\s*\d', full_text, re.I):
        return 'INR'
    if '$' in full_text or re.search(r'\bUSD\b', full_text, re.I):
        return 'USD'
    if '€' in full_text or re.search(r'\bEUR\b', full_text, re.I):
        return 'EUR'
    if '£' in full_text or re.search(r'\bGBP\b', full_text, re.I):
        return 'GBP'
    return None


def extract_labeled_amount(lines, keywords):
    # Exact high-priority labels first; amount must be on the same line.
    for line in lines:
        text = line.get('text', '')
        if any(re.search(k, text, re.I) for k in keywords):
            vals = amounts_in_line(text, allow_bare_integer=True)
            if vals:
                return vals[-1][1], text
    return None, None


def extract_tax_total(lines):
    """Sum separate tax components when present; otherwise return the best tax line."""
    components = []
    generic_tax = []
    for line in lines:
        text = line.get('text', '')
        if re.search(r'(?i)\b(?:cgst|sgst|igst|vat|sales tax)\b', text):
            vals = amounts_in_line(text, allow_bare_integer=True)
            if vals:
                components.append(vals[-1][1])
        elif re.search(r'(?i)\b(?:gst|tax)\b', text) and not re.search(r'(?i)\b(?:gstin|tax invoice|taxable value)\b', text):
            vals = amounts_in_line(text, allow_bare_integer=True)
            if vals:
                generic_tax.append(vals[-1][1])
    if components:
        return round(sum(components), 2), 'sum of tax components'
    if generic_tax:
        return generic_tax[-1], 'tax line'
    return None, None


def extract_line_items(lines):
    """Conservative candidate extraction: require an amount and reject headers/addresses."""
    items = []
    seen = set()
    for line in lines:
        text = normalize_space(line.get('text', ''))
        if len(text) < 4 or EXCLUDE_ITEM_RE.search(text):
            continue
        if find_date(text) or ADDRESS_HINT_RE.search(text):
            continue
        # Phone numbers and ZIP-like strings should not create line items.
        if re.search(r'(?i)\b(?:phone|tel|mobile|contact)\b|\b\d{5,}\b', text):
            continue

        vals = amounts_in_line(text, allow_bare_integer=False)
        if not vals:
            continue
        # Reject lines that are almost entirely numeric/currency punctuation.
        desc = MONEY_TOKEN_RE.sub(' ', text)
        desc = re.sub(r'\s+', ' ', desc).strip(' :-|')
        if len(re.sub(r'[^A-Za-z]', '', desc)) < 2:
            continue

        # A product line normally has one or more explicit monetary values.
        # With two amounts, use the last as line total and previous as unit price.
        line_total = vals[-1][1]
        unit_price = vals[-2][1] if len(vals) >= 2 else None
        qty = None
        qty_match = re.search(r'\b(\d{1,3})\s+(?:x\s*)?(?:₹|Rs\.?|INR|USD|\$|€|£)?\s*\d[\d,]*\.\d{1,2}\b', text, re.I)
        if qty_match:
            q = int(qty_match.group(1))
            if 0 < q < 10000:
                qty = q

        key = (desc.casefold(), line_total)
        if key in seen:
            continue
        seen.add(key)
        items.append({
            'description': desc,
            'quantity': qty,
            'unit_price': unit_price,
            'line_total': line_total,
            'source_text': text,
            'ocr_confidence': line.get('conf'),
            'classifier_label': line.get('classifier_label'),
            'classifier_confidence': line.get('classifier_confidence')
        })
    return items

### Parser fix: stricter monetary values and line-item filtering

The revised rules ignore bare integers as money (so dates, phone numbers, ZIP codes, and street numbers are less likely to become prices), require explicit document-number labels, reject common address/footer/total lines from line items, and sum separate CGST/SGST/IGST-style components where detected. After replacing this notebook, rerun all cells from Cell 8 onward; for a clean state, use Runtime → Restart session and run all.

## Cell 9 — Parse a document into structured data

The output includes page-level OCR text, extracted fields, line-item candidates, and warnings. A document can have multiple pages; this starter implementation combines the page lines before field extraction.

In [ ]:
def parse_document(path):
    path = Path(path)
    page_records = []
    all_lines = []

    try:
        pages = load_document_pages(path)
    except Exception as e:
        return {
            'file_name': path.name, 'status': 'error', 'error': str(e),
            'fields': {}, 'line_items': [], 'warnings': ['Could not open document.']
        }

    for page_index, (page_label, page_image) in enumerate(pages, start=1):
        try:
            words = ocr_page(page_image, preprocess=PREPROCESS_FOR_OCR)
            lines = classify_lines(words_to_lines(words))
            for line in lines:
                line['page'] = page_index
            all_lines.extend(lines)
            page_records.append({
                'page': page_index, 'label': page_label,
                'text': '\n'.join(line['text'] for line in lines),
                'line_count': len(lines)
            })
        except Exception as e:
            page_records.append({'page': page_index, 'label': page_label, 'text': '', 'error': str(e)})

    full_text = '\n'.join(line['text'] for line in all_lines)

    # Vendor heuristic: look only at the first several lines and avoid address/date/document labels.
    vendor = None
    vendor_exclusions = re.compile(
        r'(?i)invoice|receipt|tax invoice|bill|phone|tel|www\.|https?://|email|gstin|'
        r'address|date|received|payment|total|subtotal|ship to|bill to|reminders?'
    )
    email_re = re.compile(r'(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b')
    top_lines = sorted(all_lines, key=lambda x: (x.get('page', 1), x.get('top', 0)))[:8]
    vendor_candidates = []
    for line in top_lines:
        txt = normalize_space(line.get('text', ''))
        letters = re.sub(r'[^A-Za-z]', '', txt)
        if (len(letters) >= 4 and len(txt) <= 70 and not vendor_exclusions.search(txt)
                and not find_date(txt) and not ADDRESS_HINT_RE.search(txt)
                and not email_re.search(txt) and '@' not in txt
                and not re.search(r'(?i)\b(?:gmail|yahoo|outlook|hotmail)\.com\b', txt)
                and not re.search(r'\b\d{4,}\b', txt)):
            # Prefer short, name-like lines with a high alphabetic proportion.
            alpha_ratio = len(letters) / max(1, len(txt))
            score = alpha_ratio - (0.15 if len(txt) > 45 else 0)
            vendor_candidates.append((score, line))
    if vendor_candidates:
        vendor = max(vendor_candidates, key=lambda x: x[0])[1]['text']

    doc_number = extract_document_number(all_lines)
    date = None
    for line in all_lines:
        # Avoid treating arbitrary dates in footer text as document dates; prefer labeled dates.
        if re.search(r'(?i)\b(?:invoice date|date|received|issued)\b', line['text']):
            date = find_date(line['text'])
            if date:
                break
    if not date:
        for line in all_lines[:8]:
            date = find_date(line['text'])
            if date:
                break

    subtotal, subtotal_source = extract_labeled_amount(
        all_lines, [r'\bsubtotal\b', r'sub\s*total', r'net amount', r'taxable value']
    )
    tax, tax_source = extract_tax_total(all_lines)

    # Prefer most specific total labels, and only then generic TOTAL.
    total, total_source = extract_labeled_amount(
        all_lines, [r'grand\s*total', r'amount\s*due', r'balance\s*due', r'amount\s*payable']
    )
    if total is None:
        # Generic TOTAL must be a financial label; amounts_in_line ignores bare integers by default.
        total, total_source = extract_labeled_amount(all_lines, [r'(?i)^\s*total\b', r'\btotal\s*[:$]'])

    gstin_match = GSTIN_RE.search(full_text.upper())
    currency = detect_currency(full_text)
    line_items = extract_line_items(all_lines)

    warnings_list = []
    if not full_text.strip():
        warnings_list.append('No text detected by OCR.')
    if not vendor:
        warnings_list.append('Vendor/merchant name not confidently identified; review manually.')
    if not doc_number:
        warnings_list.append('Document number not detected; verify manually.')
    if not date:
        warnings_list.append('Date not detected; verify manually.')
    if total is None:
        warnings_list.append('Grand total not detected; verify manually.')
    if not line_items:
        warnings_list.append('No line-item candidates detected.')
    if any(line.get('conf', 100) < 45 for line in all_lines):
        warnings_list.append('Some OCR lines have low confidence.')
    # Warn when the classifier is uncertain instead of using it as an authoritative decision.
    if any(line.get('classifier_confidence', 1) < 0.35 for line in all_lines):
        warnings_list.append('Classifier confidence is low on some lines; rules and manual review are important.')
    if total is not None and subtotal is not None and tax is not None:
        if abs((subtotal + tax) - total) > max(2.0, abs(total) * 0.02):
            warnings_list.append('Subtotal + tax does not reconcile with total; check discounts, fees, rounding, or OCR.')

    fields = {
        'vendor': vendor,
        'document_number': doc_number,
        'date': date,
        'currency': currency,
        'subtotal': subtotal,
        'tax': tax,
        'grand_total': total,
        'gstin': gstin_match.group(0).upper() if gstin_match else None,
        'subtotal_source_text': subtotal_source,
        'tax_source_text': tax_source,
        'total_source_text': total_source
    }
    return {
        'file_name': path.name,
        'status': 'ok' if full_text.strip() else 'needs_review',
        'document_type': 'invoice_or_receipt',
        'fields': fields,
        'line_items': line_items,
        'pages': page_records,
        'ocr_lines': all_lines,
        'full_text': full_text,
        'warnings': warnings_list
    }

## Cell 10 — Regression test — vendor and document number fixes

This test checks the two reported failure modes: email addresses must not be selected as vendors, and a document number on the line after `Invoice number:` must be found. It is a rule-level test, separate from OCR accuracy.

In [ ]:
# Targeted tests for the reported receipt layout.
test_lines = [
    {"text": "GENERIC COMPANY", "top": 10, "page": 1},
    {"text": "company@abc.com", "top": 35, "page": 1},
    {"text": "Invoice number:", "top": 60, "page": 1},
    {"text": "5177183057228876365", "top": 85, "page": 1},
]
assert extract_document_number(test_lines) == "5177183057228876365", extract_document_number(test_lines)

email_re = re.compile(r'(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b')
assert email_re.search("company@abc.com")
print("PASS: adjacent-line document number extraction works.")
print("PASS: email addresses can be explicitly excluded from vendor candidates.")

## Cell 11 — Run the parser on uploaded documents

This produces one result per file. Each PDF is one document, even if it has several pages.

In [ ]:
results = []
for path in input_files:
    print("Parsing:", path.name)
    result = parse_document(path)
    results.append(result)
    print("  status:", result.get("status"))
    print("  vendor:", result.get("fields", {}).get("vendor"))
    print("  total:", result.get("fields", {}).get("grand_total"))
    print("  warnings:", result.get("warnings", []))

print(f"\nParsed {len(results)} document(s).")

## Cell 12 — Synthetic sample test (no upload required)

Use this test to check rule behavior without a physical document. It does **not** test OCR quality; it tests the extraction rules against clean text.

In [ ]:
SAMPLE_TEXT = [
    "ABC OFFICE SUPPLIES",
    "TAX INVOICE",
    "Invoice No: INV-2026-1042",
    "Date: 12/10/2026",
    "GSTIN: 22AAAAA0000A1Z5",
    "Printer paper A4 2 500.00 1000.00",
    "Subtotal 1,000.00",
    "CGST 90.00",
    "SGST 90.00",
    "Grand Total INR 1,180.00",
]

sample_lines = []
for i, text in enumerate(SAMPLE_TEXT):
    sample_lines.append({
        "text": text, "conf": 95.0, "left": 0, "top": i * 30,
        "right": 600, "bottom": i * 30 + 20, "words": text.split(), "page": 1
    })
sample_lines = classify_lines(sample_lines)

sample_full_text = "\n".join(x["text"] for x in sample_lines)
sample_fields = {
    "document_number": extract_document_number(sample_lines),
    "date": next((find_date(x["text"]) for x in sample_lines if find_date(x["text"])), None),
    "currency": detect_currency(sample_full_text),
    "subtotal": extract_labeled_amount(sample_lines, [r'\bsubtotal\b', r'sub\s*total', r'net amount', r'taxable value'])[0],
    "tax": extract_labeled_amount(sample_lines, [r'\b(?:gst|cgst|sgst|igst|vat|sales tax)\b'])[0],
    "grand_total": extract_labeled_amount(sample_lines, [r'grand total', r'amount due', r'balance due', r'amount payable', r'\btotal\b'])[0],
    "gstin": (GSTIN_RE.search(sample_full_text.upper()).group(0).upper() if GSTIN_RE.search(sample_full_text.upper()) else None)
}
sample_items = extract_line_items(sample_lines)
print(json.dumps({"fields": sample_fields, "line_items": sample_items}, indent=2, ensure_ascii=False))

## Cell 13 — Validate extraction results

These checks flag missing fields and arithmetic mismatches. They cannot prove that a field is correct; compare important values against the source document.

In [ ]:
def validate_result(result):
    fields = result.get("fields", {})
    warnings_out = list(result.get("warnings", []))
    required = ["vendor", "grand_total"]
    missing = [name for name in required if fields.get(name) is None]
    if missing:
        warnings_out.append("Required field(s) missing: " + ", ".join(missing))

    subtotal = fields.get("subtotal")
    tax = fields.get("tax")
    total = fields.get("grand_total")
    if subtotal is not None and tax is not None and total is not None:
        diff = round((subtotal + tax) - total, 2)
        if abs(diff) > max(2.0, abs(total) * 0.02):
            warnings_out.append(f"Arithmetic mismatch: subtotal + tax - total = {diff:.2f}")

    return {
        "file_name": result.get("file_name"),
        "status": result.get("status"),
        "vendor": fields.get("vendor"),
        "document_number": fields.get("document_number"),
        "date": fields.get("date"),
        "currency": fields.get("currency"),
        "subtotal": subtotal,
        "tax": tax,
        "grand_total": total,
        "gstin": fields.get("gstin"),
        "line_item_count": len(result.get("line_items", [])),
        "warning_count": len(set(warnings_out)),
        "warnings": " | ".join(dict.fromkeys(warnings_out))
    }

validation_rows = [validate_result(r) for r in results]
validation_df = pd.DataFrame(validation_rows)
display(validation_df if not validation_df.empty else pd.DataFrame(columns=[
    "file_name", "status", "vendor", "document_number", "date", "currency",
    "subtotal", "tax", "grand_total", "gstin", "line_item_count", "warning_count", "warnings"
]))

## Cell 14 — Evaluate against manually labeled ground truth

Create a small CSV with one row per document and the correct values. Use the same filename as the uploaded file. This cell computes exact-match accuracy for selected fields. For numeric amounts, exact match is strict; you can change the comparison to a tolerance if your data requires it.

In [ ]:
# Example ground-truth template. Edit the values and save it as /content/ground_truth.csv.
GROUND_TRUTH_TEMPLATE = pd.DataFrame([
    {
        "file_name": "example_receipt.jpg",
        "vendor": "Example Store",
        "document_number": "R12345",
        "date": "2026-10-12",
        "grand_total": 1180.00
    }
])
GROUND_TRUTH_TEMPLATE.to_csv("/content/ground_truth_template.csv", index=False)
print("Created /content/ground_truth_template.csv. Download and edit it, then upload/save it as /content/ground_truth.csv.")

def evaluate_against_ground_truth(results, ground_truth_path="/content/ground_truth.csv"):
    gt_path = Path(ground_truth_path)
    if not gt_path.exists():
        print("Ground truth file not found. Create it from /content/ground_truth_template.csv.")
        return None
    gt = pd.read_csv(gt_path).fillna("")
    predicted = []
    for result in results:
        f = result.get("fields", {})
        predicted.append({
            "file_name": result.get("file_name"),
            "vendor": f.get("vendor") or "",
            "document_number": f.get("document_number") or "",
            "date": f.get("date") or "",
            "grand_total": f.get("grand_total")
        })
    pred = pd.DataFrame(predicted)
    merged = gt.merge(pred, on="file_name", how="inner", suffixes=("_true", "_pred"))
    if merged.empty:
        print("No matching filenames between ground truth and parsed results.")
        return None

    metrics = {}
    for field in ["vendor", "document_number", "date"]:
        true = merged[f"{field}_true"].astype(str).str.strip().str.casefold()
        predv = merged[f"{field}_pred"].fillna("").astype(str).str.strip().str.casefold()
        metrics[field + "_exact_match_accuracy"] = float((true == predv).mean())

    numeric_rows = merged.dropna(subset=["grand_total_true", "grand_total_pred"])
    if len(numeric_rows):
        metrics["grand_total_exact_match_accuracy"] = float(
            (pd.to_numeric(numeric_rows["grand_total_true"], errors="coerce").round(2) ==
             pd.to_numeric(numeric_rows["grand_total_pred"], errors="coerce").round(2)).mean()
        )
        metrics["grand_total_within_1_unit_accuracy"] = float(
            (pd.to_numeric(numeric_rows["grand_total_true"], errors="coerce") -
             pd.to_numeric(numeric_rows["grand_total_pred"], errors="coerce")).abs().le(1.0).mean()
        )
    metrics["documents_evaluated"] = int(len(merged))
    return metrics, merged

# Run after preparing /content/ground_truth.csv:
evaluation = evaluate_against_ground_truth(results)
if evaluation:
    metrics, comparison = evaluation
    print(json.dumps(metrics, indent=2))
    display(comparison)

## Cell 15 — Export JSON, CSV, and Excel

Exports:
- `parsed_documents.json`: complete results, including OCR lines and line-item candidates;
- `documents.csv`: one row per document;
- `line_items.csv`: one row per extracted line-item candidate;
- `receipt_invoice_results.xlsx`: summary, line items, and warnings sheets.

The JSON is the most complete output. CSV and Excel contain convenient flattened tables.

In [ ]:
def json_safe(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

json_path = OUTPUT_DIR / "parsed_documents.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=json_safe)

document_rows = []
item_rows = []
warning_rows = []

for result in results:
    fields = result.get("fields", {})
    document_rows.append({
        "file_name": result.get("file_name"),
        "status": result.get("status"),
        "vendor": fields.get("vendor"),
        "document_number": fields.get("document_number"),
        "date": fields.get("date"),
        "currency": fields.get("currency"),
        "subtotal": fields.get("subtotal"),
        "tax": fields.get("tax"),
        "grand_total": fields.get("grand_total"),
        "gstin": fields.get("gstin"),
        "line_item_count": len(result.get("line_items", [])),
        "warnings": " | ".join(result.get("warnings", []))
    })
    for idx, item in enumerate(result.get("line_items", []), start=1):
        item_rows.append({"file_name": result.get("file_name"), "item_number": idx, **item})
    for warning in result.get("warnings", []):
        warning_rows.append({"file_name": result.get("file_name"), "warning": warning})

documents_df = pd.DataFrame(document_rows)
items_df = pd.DataFrame(item_rows)
warnings_df = pd.DataFrame(warning_rows)

documents_csv = OUTPUT_DIR / "documents.csv"
items_csv = OUTPUT_DIR / "line_items.csv"
xlsx_path = OUTPUT_DIR / "receipt_invoice_results.xlsx"

documents_df.to_csv(documents_csv, index=False)
items_df.to_csv(items_csv, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    documents_df.to_excel(writer, sheet_name="Documents", index=False)
    items_df.to_excel(writer, sheet_name="Line Items", index=False)
    warnings_df.to_excel(writer, sheet_name="Warnings", index=False)

print("Created:")
for p in [json_path, documents_csv, items_csv, xlsx_path]:
    print(p, f"({p.stat().st_size:,} bytes)")

## Cell 16 — Download outputs

Download the main Excel workbook, CSV files, and full JSON. Run the export cell first.

In [ ]:
from google.colab import files

for path in [
    OUTPUT_DIR / "receipt_invoice_results.xlsx",
    OUTPUT_DIR / "documents.csv",
    OUTPUT_DIR / "line_items.csv",
    OUTPUT_DIR / "parsed_documents.json"
]:
    if path.exists():
        files.download(str(path))
    else:
        print("Missing output:", path)

## Cell 17 — Improve the classifier with your own labels

For better classification, add real OCR lines paired with their correct labels. Use labels such as `vendor`, `document_number`, `date`, `subtotal`, `tax`, `total`, `line_item`, and `other`. Keep examples from multiple vendors, and split evaluation data by vendor/document rather than randomly splitting lines from the same invoice.

Run this cell after editing `CUSTOM_TRAINING_EXAMPLES`; it retrains the lightweight classifier.

In [ ]:
CUSTOM_TRAINING_EXAMPLES = [
    # Add examples from your own documents:
    # ("YOUR MERCHANT NAME", "vendor"),
    # ("Invoice No: ABC-123", "document_number"),
    # ("Grand Total 2450.00", "total"),
    # ("CGST 125.00", "tax"),
    # ("Ballpoint pens 4 20.00 80.00", "line_item"),
]

if CUSTOM_TRAINING_EXAMPLES:
    combined_examples = TRAINING_EXAMPLES + CUSTOM_TRAINING_EXAMPLES
    classifier.fit([x for x, y in combined_examples], [y for x, y in combined_examples])
    print("Retrained classifier with", len(combined_examples), "examples.")
else:
    print("No custom examples added; starter classifier unchanged.")

## Cell 18 — Practical limitations and next improvements

1. **Vendor names:** the first non-generic line near the top is only a heuristic; logos, addresses, and headings can confuse it.
2. **Dates:** `12/10/2026` can mean 12 October or 10 December depending on locale. Normalize dates only after defining your expected locale.
3. **Line items:** this baseline finds candidate rows, not guaranteed correct table columns. Tables with wrapped descriptions or multi-line tax columns need layout-specific logic.
4. **Totals:** `TOTAL` can appear in several contexts. Explicit labels are preferred, but always review extracted values.
5. **Classifier:** starter examples are illustrative, not a production training set. A small classifier helps organize lines but cannot reliably infer every layout.
6. **Evaluation:** collect representative labeled receipts and invoices, measure field-level accuracy and line-item accuracy, and keep a separate test set.
7. **Privacy:** uploaded files remain in the Colab runtime for the session. Avoid uploading sensitive financial documents to shared notebooks, and delete runtime files when finished.
8. **Production readiness:** add duplicate detection, human review/editing, audit logs, currency-specific number parsing, and robust multi-page table reconstruction before relying on it for bookkeeping.